# 🧠 3D Spatial Reasoning RLHF — Validation with Real ScanNet Data

**Approach A: Multi-View Rendering → VLM**

```
ScanNet 3D Scans → Extract RGB frames (multiple camera views)
                 → Feed multi-view images to Qwen2.5-VL
                 → VLM generates spatial reasoning Q&A
                 → RLHF preference pairs (strong CoT vs weak shallow)
```

**What's happening technically:**
- ScanNet provides RGB-D video of real 3D-scanned rooms (968 scenes)
- We sample N diverse camera viewpoints from each scene's video frames
- We pass these as multi-image input to Qwen2.5-VL along with scene metadata (object labels, 3D bounding boxes)
- The VLM reasons about spatial relations, occlusion, counting from the rendered views

⚠️ **Runtime**: Use **T4** (4-bit quant) or **A100** (float16). Select GPU in Runtime → Change runtime type.

## 0. Install Dependencies

In [ ]:
# First verify GPU
!nvidia-smi

# Install deps
!pip install -q transformers>=4.45.0 accelerate bitsandbytes
!pip install -q qwen-vl-utils Pillow requests tqdm
!pip install -q datasets huggingface_hub pandas

In [ ]:
import torch
import json
import os
import re
import textwrap
from pathlib import Path
from datetime import datetime
from collections import Counter
from PIL import Image
from io import BytesIO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load Real ScanNet Data from HuggingFace

We use two datasets:
- **`ZiAngGu/scannet_3dbox_v2`** — 33k rows with RGB images + 3D bounding boxes + object descriptions from ScanNet
- Each row has a real RGB frame from a ScanNet scene with annotated 3D objects

This gives us **real room images** with **ground-truth 3D metadata** — exactly what we need.

In [ ]:
from datasets import load_dataset

# Load ScanNet 3D box dataset — has RGB images + 3D bounding boxes + descriptions
# This downloads ~2.8GB; streaming=True to avoid full download
print("Loading ScanNet dataset from HuggingFace (streaming)...")
ds = load_dataset("ZiAngGu/scannet_3dbox_v2", split="train", streaming=True)

# Grab first 200 samples to work with (covers multiple scenes)
raw_samples = []
for i, sample in enumerate(ds):
    raw_samples.append(sample)
    if i >= 199:
        break

print(f"Loaded {len(raw_samples)} samples")
print(f"Columns: {list(raw_samples[0].keys())}")

# Preview one sample
s = raw_samples[0]
print(f"\nSample keys: {list(s.keys())}")
for k, v in s.items():
    if isinstance(v, Image.Image):
        print(f"  {k}: PIL Image {v.size}")
    elif isinstance(v, str):
        print(f"  {k}: {v[:120]}...")
    else:
        print(f"  {k}: {type(v).__name__} = {str(v)[:120]}")

In [ ]:
# Group samples by scene and build scene-level data with multiple views
from collections import defaultdict

scenes_data = defaultdict(lambda: {"images": [], "descriptions": [], "metadata": {}})

for sample in raw_samples:
    # The dataset has image + text description columns
    # Identify scene from the data — inspect actual column names
    # We'll group by description patterns or just use sequential grouping
    desc = sample.get("text", sample.get("description", ""))
    img = sample.get("image", None)

    if img is not None:
        # Use a simple scene ID based on description similarity
        # Each unique major description = different scene viewpoint
        scene_key = desc[:80] if desc else f"scene_{len(scenes_data)}"

        # Try to find the actual scene identifier
        for k, v in sample.items():
            if k not in ("image", "text", "description") and isinstance(v, str):
                if "scene" in v.lower() or "scan" in v.lower():
                    scene_key = v
                    break

        scenes_data[scene_key]["images"].append(img)
        scenes_data[scene_key]["descriptions"].append(desc)

print(f"Found {len(scenes_data)} unique scene groups")
for k in list(scenes_data.keys())[:5]:
    sd = scenes_data[k]
    print(f"  '{k[:60]}': {len(sd['images'])} views")

# If grouping didn't work well (all unique keys), just batch every 5 images as a "scene"
if len(scenes_data) > 50:
    print("\nRe-grouping: batching every 5 consecutive frames as a scene view set...")
    scenes_data = {}
    for i in range(0, min(len(raw_samples), 100), 5):
        batch = raw_samples[i:i+5]
        scene_id = f"scannet_scene_{i//5:03d}"
        scenes_data[scene_id] = {
            "images": [s.get("image") for s in batch if s.get("image") is not None],
            "descriptions": [s.get("text", s.get("description", "")) for s in batch],
        }
    print(f"Created {len(scenes_data)} scene groups with ~5 views each")

# Preview images from first scene
first_key = list(scenes_data.keys())[0]
first_scene = scenes_data[first_key]
print(f"\nFirst scene '{first_key}': {len(first_scene['images'])} views")
print(f"Description: {first_scene['descriptions'][0][:200]}")

# Display first view
first_scene["images"][0]

## 2. Load Qwen2.5-VL-7B

This VLM can process **multiple images** in a single prompt — perfect for multi-view scene understanding. We feed it N camera views of a room and ask spatial reasoning questions.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Use 3B for T4 (15GB), 7B for A100 (40/80GB)
if gpu_mem_gb >= 40:
    MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
    print(f"A100 detected ({gpu_mem_gb:.0f}GB) → using 7B model")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
else:
    MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
    print(f"T4 detected ({gpu_mem_gb:.0f}GB) → using 3B model in 4-bit")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant_config, device_map="auto"
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)
print(f"✅ {MODEL_ID} loaded")

In [ ]:
IMG_SIZE = 480 if gpu_mem_gb < 40 else 720

def query_qwen_vl(system_prompt: str, user_text: str, images: list = None, max_tokens: int = 2048, temperature: float = 0.7) -> str:
    """Query Qwen2.5-VL with optional multi-image input."""
    user_content = []
    if images:
        for img in images:
            img_resized = img.copy()
            img_resized.thumbnail((IMG_SIZE, IMG_SIZE))
            user_content.append({"type": "image", "image": img_resized})
        user_content.append({"type": "text", "text": f"[The above are {len(images)} different camera views of the same 3D scene.]\n\n{user_text}"})
    else:
        user_content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": user_content},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if images:
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                          return_tensors="pt", padding=True).to(model.device)
    else:
        inputs = processor(text=[text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_tokens,
                                     temperature=temperature, do_sample=True)

    generated = output_ids[0][inputs.input_ids.shape[1]:]
    torch.cuda.empty_cache()
    return processor.decode(generated, skip_special_tokens=True)

print(f"Image size: {IMG_SIZE}px")

# Sanity check — text only
test = query_qwen_vl(
    "You are a spatial reasoning assistant.",
    "A cup is at [0.1, 0.0, 0.78] and a plate is at [0.0, 0.0, 0.78]. Which is to the right?",
    max_tokens=100
)
print(f"✅ Text-only: {test[:200]}")

# Sanity check — with an image
first_scene = scenes_data[list(scenes_data.keys())[0]]
if first_scene["images"]:
    test_img = query_qwen_vl(
        "Describe what you see.",
        "What objects can you identify? List them briefly.",
        images=first_scene["images"][:1],
        max_tokens=150
    )
    print(f"✅ Image: {test_img[:300]}")

## 3. Generate Spatial Reasoning Questions from Real Scene Images

Feed multi-view ScanNet images + scene descriptions to the VLM. It generates diverse questions about spatial relations, counting, occlusion, and robot manipulation.

In [ ]:
QUESTION_GEN_SYSTEM = """Generate spatial reasoning questions about 3D scenes for robotics RLHF. Output ONLY valid JSON."""

QUESTION_GEN_USER = """Scene: {descriptions}

Generate {n} questions across these categories: counting, spatial, occlusion, affordance, manipulation, scene.
Vary difficulty: easy, medium, hard.

Output JSON:
{{"questions": [{{"id": 1, "text": "...", "category": "...", "difficulty": "..."}}]}}"""

MAX_VIEWS = 2 if gpu_mem_gb < 24 else 4

def generate_questions_for_scene(scene_id: str, scene_data: dict, n: int = 10) -> list:
    """Generate questions using real scene images + descriptions."""
    descriptions = "; ".join(d[:100] for d in scene_data["descriptions"] if d)
    prompt = QUESTION_GEN_USER.format(descriptions=descriptions, n=n)

    images = scene_data["images"][:MAX_VIEWS]

    response = query_qwen_vl(QUESTION_GEN_SYSTEM, prompt, images=images, max_tokens=1500)
    torch.cuda.empty_cache()

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', response)
        if match:
            try:
                data = json.loads(match.group())
            except:
                print(f"  ⚠️ Failed to parse JSON for {scene_id}")
                print(f"  Raw: {response[:300]}")
                return []
        else:
            print(f"  ⚠️ No JSON found for {scene_id}")
            print(f"  Raw: {response[:300]}")
            return []

    questions = data.get("questions", [])
    for q in questions:
        q["scene_id"] = scene_id
    return questions

print(f"Using {MAX_VIEWS} views per scene (GPU: {gpu_mem_gb:.0f}GB)")

In [ ]:
from tqdm import tqdm

NUM_SCENES = 5
QUESTIONS_PER_SCENE = 10

scene_ids = list(scenes_data.keys())[:NUM_SCENES]
all_questions = []

for scene_id in tqdm(scene_ids, desc="Generating questions"):
    scene = scenes_data[scene_id]
    if not scene["images"]:
        continue

    qs = generate_questions_for_scene(scene_id, scene, n=QUESTIONS_PER_SCENE)
    all_questions.extend(qs)

    print(f"\n{scene_id}: {len(qs)} questions generated")
    for q in qs[:3]:
        print(f"  [{q.get('category','?')}/{q.get('difficulty','?')}] {q['text']}")

print(f"\n{'='*50}")
print(f"Total: {len(all_questions)} questions across {len(scene_ids)} scenes")

with open(OUTPUT_DIR / "questions.json", "w") as f:
    json.dump(all_questions, f, indent=2)
print(f"Saved to {OUTPUT_DIR / 'questions.json'}")

## 4. Generate RLHF Answer Pairs — Strong (Chosen) vs Weak (Rejected)

Key difference for RLHF:
- **Chosen (strong)**: Gets the images + full scene context, uses CoT reasoning, low temperature → accurate spatial answer
- **Rejected (weak)**: Gets NO images, only a brief text description, high temperature → shallow/wrong answer

This naturally creates the preference gap we need for RLHF training.

In [ ]:
ANSWER_STRONG_SYSTEM = """You are a spatial reasoning VLM for robotics. You analyze 3D scenes from multiple camera views and answer questions with precise step-by-step reasoning.

Always follow Chain-of-Thought:
1. **Observe**: Describe what you see in the images relevant to the question
2. **Reason**: Apply spatial logic — consider what's visible, occluded, reachable
3. **Answer**: Give a clear, precise final answer

Be accurate. If something is partially occluded or uncertain from the views, say so."""

ANSWER_STRONG_USER = """Scene annotations: {descriptions}

Question: {question}

Look at the provided camera views carefully and respond with step-by-step spatial reasoning (Observe → Reason → Answer)."""

ANSWER_WEAK_SYSTEM = """Answer questions about rooms briefly and directly. Don't overthink it."""

ANSWER_WEAK_USER = """Room with objects: {descriptions_brief}
Question: {question}
Answer:"""


def gen_answer_strong(scene_data: dict, question: str) -> str:
    """Strong answer: gets images + full context, CoT, low temp."""
    descriptions = "; ".join(d for d in scene_data["descriptions"][:3] if d)
    prompt = ANSWER_STRONG_USER.format(descriptions=descriptions, question=question)
    images = scene_data["images"][:3]
    return query_qwen_vl(ANSWER_STRONG_SYSTEM, prompt, images=images,
                         max_tokens=600, temperature=0.3)


def gen_answer_weak(scene_data: dict, question: str) -> str:
    """Weak answer: NO images, minimal context, high temp → shallow/inaccurate."""
    descriptions_brief = "; ".join(d[:60] for d in scene_data["descriptions"][:2] if d)
    prompt = ANSWER_WEAK_USER.format(descriptions_brief=descriptions_brief, question=question)
    # Intentionally NO images — forces hallucination/guessing
    return query_qwen_vl(ANSWER_WEAK_SYSTEM, prompt, images=None,
                         max_tokens=150, temperature=0.9)

In [ ]:
import base64
from io import BytesIO

def images_to_base64(images, max_size=480):
    """Convert PIL images to base64 data URIs for export."""
    encoded = []
    for img in images:
        thumb = img.copy()
        thumb.thumbnail((max_size, max_size))
        buf = BytesIO()
        thumb.save(buf, format="JPEG", quality=75)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        encoded.append(f"data:image/jpeg;base64,{b64}")
    return encoded

# Generate RLHF pairs — this is the expensive step (~2 LLM calls per question)
rlhf_pairs = []

for q in tqdm(all_questions, desc="Generating answer pairs"):
    scene_id = q["scene_id"]
    scene = scenes_data[scene_id]
    question = q["text"]

    try:
        chosen = gen_answer_strong(scene, question)
        rejected = gen_answer_weak(scene, question)

        # Encode the images the model actually saw
        view_images = scene["images"][:3]
        image_b64s = images_to_base64(view_images, max_size=IMG_SIZE)

        # Find which raw_sample indices belong to this scene
        row_indices = [i for i, s in enumerate(raw_samples)
                       if scenes_data.get(scene_id) and s.get("image") in scene["images"]]

        pair = {
            "prompt": f"User: A robot is observing a room. {question}",
            "chosen": chosen,
            "rejected": rejected,
            "scene_id": scene_id,
            "category": q.get("category", ""),
            "difficulty": q.get("difficulty", ""),

            # Source lineage — trace back to exact dataset rows
            "source": {
                "dataset": "ZiAngGu/scannet_3dbox_v2",
                "split": "train",
                "row_indices": row_indices,
                "scene_id": scene_id,
                "images": image_b64s,
            },

            # Generation metadata — how was this pair made?
            "generation": {
                "model": MODEL_ID,
                "chosen_strategy": "multi-view-cot",
                "rejected_strategy": "text-only-shallow",
                "chosen_temperature": 0.3,
                "rejected_temperature": 0.9,
                "num_views": len(view_images),
                "image_resolution": IMG_SIZE,
                "generated_at": datetime.now().isoformat(),
            },
        }
        rlhf_pairs.append(pair)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print(f"\n✅ Generated {len(rlhf_pairs)} RLHF pairs")
print(f"   Each pair includes {len(rlhf_pairs[0]['source']['images'])} base64 images" if rlhf_pairs else "")

## 5. Inspect RLHF Pairs — Compare Chosen vs Rejected

In [ ]:
# Pretty-print pairs to see quality difference
for i, pair in enumerate(rlhf_pairs[:5]):
    print(f"\n{'='*70}")
    print(f"PAIR {i+1} | {pair['category']} | {pair['difficulty']} | {pair['scene_id']}")
    print(f"{'='*70}")
    print(f"\n📝 PROMPT: {pair['prompt']}")
    print(f"\n✅ CHOSEN (multi-view CoT, {len(pair['chosen'])} chars):")
    print(textwrap.indent(pair['chosen'][:600], '  '))
    if len(pair['chosen']) > 600: print("  ...")
    print(f"\n❌ REJECTED (text-only shallow, {len(pair['rejected'])} chars):")
    print(textwrap.indent(pair['rejected'][:400], '  '))
    print()

In [ ]:
import matplotlib.pyplot as plt
import textwrap as tw

def show_pair_with_images(pair_idx, pairs, scenes):
    """Show scene images + chosen/rejected answers for visual quality check."""
    pair = pairs[pair_idx]
    scene_id = pair["scene_id"]
    scene = scenes[scene_id]
    images = scene["images"][:3]

    n_imgs = len(images)
    fig, axes = plt.subplots(1, n_imgs + 1, figsize=(6 * (n_imgs + 1), 6),
                              gridspec_kw={"width_ratios": [1]*n_imgs + [2]})
    if n_imgs + 1 == 2:
        axes = [axes[0], axes[1]]

    # Show scene images
    for i, img in enumerate(images):
        axes[i].imshow(img)
        axes[i].set_title(f"View {i+1}", fontsize=12)
        axes[i].axis("off")

    # Show Q&A text
    axes[-1].axis("off")
    text = (
        f"PAIR {pair_idx+1} | {pair['category']} | {pair['difficulty']}\n"
        f"{'─'*60}\n\n"
        f"📝 QUESTION:\n{tw.fill(pair['prompt'], 70)}\n\n"
        f"✅ CHOSEN ({len(pair['chosen'])} chars):\n{tw.fill(pair['chosen'][:500], 70)}\n"
        f"{'...' if len(pair['chosen']) > 500 else ''}\n\n"
        f"❌ REJECTED ({len(pair['rejected'])} chars):\n{tw.fill(pair['rejected'][:300], 70)}"
    )
    axes[-1].text(0, 1, text, transform=axes[-1].transAxes, fontsize=9,
                  verticalalignment="top", fontfamily="monospace",
                  bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3))

    plt.tight_layout()
    plt.show()

# Show all pairs with their scene images
for i in range(len(rlhf_pairs)):
    show_pair_with_images(i, rlhf_pairs, scenes_data)

In [ ]:
# Save final RLHF dataset
with open(OUTPUT_DIR / "rlhf_pairs.json", "w") as f:
    json.dump(rlhf_pairs, f, indent=2)

cats = Counter(p["category"] for p in rlhf_pairs)
diffs = Counter(p["difficulty"] for p in rlhf_pairs)

stats = {
    "timestamp": datetime.now().isoformat(),
    "approach": "multi-view-rendering-to-vlm",
    "model": "Qwen2.5-VL-7B-Instruct",
    "data_source": "ZiAngGu/scannet_3dbox_v2 (HuggingFace)",
    "num_scenes": len(scene_ids),
    "total_questions": len(all_questions),
    "total_rlhf_pairs": len(rlhf_pairs),
    "categories": dict(cats),
    "difficulties": dict(diffs),
    "chosen_model": "qwen2.5-vl-7b with multi-view images + CoT prompting",
    "rejected_model": "qwen2.5-vl-7b text-only, no images, shallow prompting",
}

with open(OUTPUT_DIR / "stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("📊 Dataset Statistics:")
print(f"  Scenes: {len(scene_ids)}")
print(f"  Questions: {len(all_questions)}")
print(f"  RLHF pairs: {len(rlhf_pairs)}")
print(f"  Categories: {dict(cats)}")
print(f"  Difficulties: {dict(diffs)}")
print(f"\n💾 Saved to {OUTPUT_DIR}/")

## 6. Automated Quality Check

Before human review — quick checks that the chosen/rejected answers are actually differentiated.

In [ ]:
def quality_check(pairs: list) -> dict:
    """Basic automated quality metrics."""
    metrics = {
        "chosen_avg_len": 0,
        "rejected_avg_len": 0,
        "chosen_has_cot": 0,  # Contains Observe/Reason/Answer markers
        "rejected_has_cot": 0,
        "length_ratio": 0,  # chosen should be longer
        "chosen_mentions_position": 0,  # References coordinates
        "rejected_mentions_position": 0,
        "chosen_mentions_occlusion": 0,
        "rejected_mentions_occlusion": 0,
    }

    for p in pairs:
        c, r = p["chosen"], p["rejected"]
        metrics["chosen_avg_len"] += len(c)
        metrics["rejected_avg_len"] += len(r)
        if any(kw in c.lower() for kw in ["observe", "reason", "answer", "step"]):
            metrics["chosen_has_cot"] += 1
        if any(kw in r.lower() for kw in ["observe", "reason", "answer", "step"]):
            metrics["rejected_has_cot"] += 1
        if any(kw in c.lower() for kw in ["position", "[0.", "coordinates", "located at"]):
            metrics["chosen_mentions_position"] += 1
        if any(kw in r.lower() for kw in ["position", "[0.", "coordinates", "located at"]):
            metrics["rejected_mentions_position"] += 1
        if any(kw in c.lower() for kw in ["occlu", "hidden", "block", "behind", "not visible"]):
            metrics["chosen_mentions_occlusion"] += 1
        if any(kw in r.lower() for kw in ["occlu", "hidden", "block", "behind", "not visible"]):
            metrics["rejected_mentions_occlusion"] += 1

    n = len(pairs) or 1
    metrics["chosen_avg_len"] /= n
    metrics["rejected_avg_len"] /= n
    metrics["length_ratio"] = metrics["chosen_avg_len"] / max(metrics["rejected_avg_len"], 1)

    return metrics

if rlhf_pairs:
    m = quality_check(rlhf_pairs)
    print("🔍 Quality Metrics:")
    print(f"  Avg chosen length:  {m['chosen_avg_len']:.0f} chars")
    print(f"  Avg rejected length: {m['rejected_avg_len']:.0f} chars")
    print(f"  Length ratio (chosen/rejected): {m['length_ratio']:.1f}x")
    print(f"  Chosen has CoT structure: {m['chosen_has_cot']}/{len(rlhf_pairs)}")
    print(f"  Rejected has CoT structure: {m['rejected_has_cot']}/{len(rlhf_pairs)}")
    print(f"  Chosen mentions positions: {m['chosen_mentions_position']}/{len(rlhf_pairs)}")
    print(f"  Chosen mentions occlusion: {m['chosen_mentions_occlusion']}/{len(rlhf_pairs)}")
    print(f"  Rejected mentions occlusion: {m['rejected_mentions_occlusion']}/{len(rlhf_pairs)}")
else:
    print("No pairs generated yet.")

## 7. Download Results

In [ ]:
import shutil
shutil.make_archive("rlhf_scannet_validation", "zip", "output")

try:
    from google.colab import files
    files.download("rlhf_scannet_validation.zip")
    print("📥 Download started!")
except ImportError:
    print("📁 Results saved to: rlhf_scannet_validation.zip")

print("\n🎯 Next steps:")
print("  1. Review rlhf_pairs.json — are chosen answers actually better?")
print("  2. Check if questions cover good diversity of spatial reasoning")
print("  3. If quality looks good → build the human ranking UI")
print("  4. Scale up: more scenes, add Chat-Scene for true 3D backbone")

## 8. 🖥️ GPU Inference Server Mode

Instead of running the full pipeline locally, expose the loaded VLM as an HTTP API.
Your Next.js backend sends `{images, prompt, system_prompt}` → Colab returns generated text.

**How to use:**
1. Run cells 4-5 (install deps + imports) and cell 10-11 (load model + helper)
2. Run the cells below to start the Flask server with ngrok tunnel
3. Copy the ngrok URL into your Next.js app's `.env.local` as `GPU_WORKER_URL`
4. Create jobs from the UI — they'll hit this server for inference

In [ ]:
!pip install -q flask pyngrok

# ⚠️ REQUIRED: Set your ngrok auth token
# 1. Sign up free at https://dashboard.ngrok.com/signup
# 2. Copy token from https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Paste below:

from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # <-- PASTE YOUR TOKEN

In [ ]:
import threading
import base64
from flask import Flask, request, jsonify
from pyngrok import ngrok
from io import BytesIO

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    """Health check — returns model info and status."""
    return jsonify({
        "status": "ready",
        "model": MODEL_ID,
        "gpu": gpu_name if torch.cuda.is_available() else "none",
        "gpu_memory_gb": round(gpu_mem_gb, 1),
    })

@app.route("/infer", methods=["POST"])
def infer():
    """
    Run VLM inference.

    Request body:
    {
        "system_prompt": "...",
        "user_prompt": "...",
        "images": ["data:image/jpeg;base64,...", ...],  // optional
        "max_tokens": 512,
        "temperature": 0.7
    }

    Response:
    {
        "text": "generated response...",
        "model": "Qwen/Qwen2.5-VL-3B-Instruct",
        "tokens_generated": 150
    }
    """
    try:
        data = request.json
        system_prompt = data.get("system_prompt", "You are a helpful assistant.")
        user_prompt = data.get("user_prompt", "")
        image_b64s = data.get("images", [])
        max_tokens = data.get("max_tokens", 512)
        temperature = data.get("temperature", 0.7)

        # Decode base64 images to PIL
        pil_images = []
        for img_str in image_b64s:
            # Strip data URI prefix if present
            if "," in img_str:
                img_str = img_str.split(",", 1)[1]
            img_bytes = base64.b64decode(img_str)
            pil_images.append(Image.open(BytesIO(img_bytes)))

        # Run inference using the same helper function from cell 11
        result = query_qwen_vl(
            system_prompt=system_prompt,
            user_text=user_prompt,
            images=pil_images if pil_images else None,
            max_tokens=max_tokens,
            temperature=temperature,
        )

        return jsonify({
            "text": result,
            "model": MODEL_ID,
            "tokens_generated": len(result.split()),  # rough estimate
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# --- Start server with ngrok tunnel ---
# Set your ngrok auth token (get one free at https://ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # Uncomment and set this!

port = 5000
public_url = ngrok.connect(port)

print(f"\n{'='*60}")
print(f"🚀 GPU Inference Server Running!")
print(f"{'='*60}")
print(f"  Local:  http://localhost:{port}")
print(f"  Public: {public_url}")
print(f"  Model:  {MODEL_ID}")
print(f"{'='*60}")
print(f"\n📋 Set this in your Next.js .env.local:")
print(f'  GPU_WORKER_URL={public_url}')
print(f"\n🧪 Test with:")
print(f'  curl {public_url}/health')
print(f"{'='*60}\n")

# Run Flask in a background thread so the cell doesn't block
threading.Thread(
    target=lambda: app.run(port=port, use_reloader=False),
    daemon=True,
).start()